# Hybrid Quantum-Classical Density Functional Theory Workflow

This Jupyter Notebook is based on the implementation of a Hybrid Quantum-Classical Density Functional Theory pipeline. Density Functional Theory (DFT) is widely used for atomistic simulations because it balances accuracy and affordability(feasible calculations). However, classical DFT has its limits, such as the lack of an exact exchange-correlation functional and the requirement of costly $O(N^{3})$ diagonalization.

In this notebook, we lay the groundwork for a hybrid quantum-classical pipeline, utilizing active-space orbital selection and core-orbital removal. By bridging classical DFT methods with quantum algorithms like the Variational Quantum Eigensolver (VQE), we can improve the precision of localized strongly correlated subsystems or accelerate computational methods.

---
## Contents:
##### 1. Theory Behind the Calculations
##### 2. How GAMESS Works
##### 3. Code Dive-in
---

## 1. Theory Behind the Calculations

### 1.1 The Molecular Hamiltonian and the Born-Oppenheimer Approximation

Every atomistic simulation begins with the Schrödinger equation. To find the total energy of a system, we define the full molecular Hamiltonian, which incorporates the kinetic and potential energies of all particles:

$$
\hat{H} = -\frac{1}{2}\sum_{i}\nabla_{i}^{2} - \frac{1}{2}\sum^{A}(1/M^{A})\nabla^{A2} - \sum_{i}\sum^{A}Z^{A}/r_{i}^{A} + \sum_{i<j}1/r_{ij} + \sum^{A<B}Z^{A}Z^{B}/R^{AB}
$$

Managing these interacting terms simultaneously is computationally intractable. Because there is a minimum of three orders of magnitude mass difference between electrons and nuclei, electrons move much faster than nuclei for the same kinetic energy. This justifies the **Born-Oppenheimer Approximation**, allowing us to treat nuclei as static relative to electrons. 

Consequently, the nuclear kinetic energy term vanishes, and the nuclei-nuclei repulsion becomes a constant shift. This decoupling yields the simplified Electronic Hamiltonian:

$$
\hat{H}_{elec} = \hat{T} + \hat{V}_{ne} + \hat{V}_{ee}
$$

### 1.2 Density Functional Theory (DFT) Fundamentals

The electronic wavefunction ($\Psi$) depends on $3N$ spatial coordinates and cannot be directly observed. Furthermore, electrons are identical fermions; according to the Anti-Symmetry Principle, the electronic wavefunction must be antisymmetric with respect to electron exchange, often represented using Slater Determinants instead of simple Hartree Products. 

Density Functional Theory asks: how can we use a simpler variable, the electron density $n(x)$, which is a function of only 3 spatial coordinates, instead of the full 3N-dimensional wavefunction?

This is justified by the two fundamental **Hohenberg-Kohn Theorems**:
1. The ground-state electron density uniquely determines the external potential.
2. The ground-state energy $E$ from the Schrödinger equation is a unique functional of the electron density $n(x)$.

The electron density that minimizes the energy of the overall functional is the true ground-state density.

### 1.3 The Kohn-Sham Equations and Exchange-Correlation

Kohn and Sham simplified this further by replacing the interacting system with a non-interacting auxiliary system that produces the exact same density. This requires solving the single-particle **Kohn-Sham Equations**:

$$
[-\frac{1}{2}\nabla^{2} + v_{eff}(\vec{r})]\phi_{i}(\vec{r}) = \epsilon_{i}\phi_{i}(\vec{r})
$$

The effective potential $v_{eff}(\vec{r})$ dictates electron behavior and is defined as:

$$
v_{eff}(\vec{r}) = v_{ext}(\vec{r}) + \int n(\vec{r}')/|\vec{r}-\vec{r}'|d^{3}r' + v_{xc}(\vec{r})
$$

This potential includes the external nuclear attraction, the Hartree potential (classical electron-electron repulsion), and the exchange-correlation potential ($v_{xc}$), which encapsulates all unknown quantum mechanical effects. Because these equations depend circularly on the electron density they produce, they must be solved iteratively via a Self-Consistent Field (SCF) cycle until convergence is reached.

Common approximations include:
*   **LDA (Local Density Approximation):** Assumes the density is locally constant; exact for a uniform electron gas.
*   **GGA (Generalized Gradient Approximation):** Includes the density gradient ($\nabla\rho$) to account for slowly varying densities.
*   **Meta-GGA:** Adds the Laplacian (curvature) and kinetic energy density for more local information.

Solving these equations creates a circular dependency: wavefunctions require the effective potential, the potential requires the electron density, and the density requires the wavefunctions. We resolve this via the Self-Consistent Field (SCF) iterative cycle: defining a trial density, solving the equations, calculating the new density, and repeating until convergence.

### 1.4 Basis Sets

To compute these equations programmatically, we represent the molecular orbitals using basis sets. A basis set is a linear combination of basis functions used to represent the electronic wavefunction, turning the partial differential equations into manageable algebraic equations. Common types include Slater Type Orbitals (STO, e.g., STO-3G) and Gaussian Type Orbitals (GTO, e.g., 6-31G).

### 1.5 Integrating Quantum Algorithms: Active Space & VQE (To be implemented)

---

## 2. How GAMESS Works

GAMESS (General Atomic and Molecular Electronic Structure System) is used for *ab initio* quantum chemistry calculations, including Self-Consistent Field (SCF) and Density Functional Theory (DFT) calculations.

### 2.1 Breakdown of Input File (`.inp`)

A GAMESS input file is organized into distinct control groups enclosed by `$` keywords and `$END`.

### Breakdown of Input Groups

```text
+-------------------------------------------------------------------------------+
|                       GAMESS INPUT FILE STRUCTURE                             |
+-------------------------------------------------------------------------------+
| $CONTRL  --->  Calculation parameters                                         |
| $SYSTEM  --->  Memory and compute resource allocation                         |
| $BASIS   --->  Basis set definition                                           |
| $DATA    --->  Molecular title, symmetry, and atomic coordinates              |
+-------------------------------------------------------------------------------+
```
Example Input File:

```fortran
 $CONTRL 
   SCFTYP=RHF 
   DFTTYP=B3LYP 
   RUNTYP=ENERGY 
   COORD=UNIQUE 
   UNITS=ANGS 
 $END

 $SYSTEM 
   MWORDS=50 
 $END

 $BASIS 
   GBASIS=N31 
   NGAUSS=6 
 $END

 $DATA
Water
C1
O   8.0   0.00000   0.00000   0.11700
H   1.0   0.00000   0.75700  -0.46900
H   1.0   0.00000  -0.75700  -0.46900
 $END
```

### 2.2 Iterative SCF Map

<pre>
                  +--------------------------------+
                  |  Molecular Structure & Basis   |
                  |  (Coordinates, Charge, 6-31G)  |
                  +---------------+----------------+
                                  |
                                  v
                  +--------------------------------+
                  |     Initial Density Guess      |
                  +---------------+----------------+
                                  |
            +--------------------->---------------------+
            |                                           |
            |                                           v
            |                         +-----------------------------------+
            |                         | Compute Potentials:               |
            |                         |  - Coulomb Potential (V_H)        |
            |                         |  - B3LYP XC-Potential (V_xc)      |
            |                         +-----------------+-----------------+
            |                                           |
            |                                           v
            |                         +-----------------------------------+
            |                         | Form Effective KS Hamiltonian     |
            |                         +-----------------+-----------------+
            |                                           |
            |                                           v
            |                         +-----------------------------------+
            |                         | Solve Kohn-Sham Equations         |
            |                         +-----------------+-----------------+
            |                                           |
            |                                           v
            |                         +-----------------------------------+
            |                         | Generate New Orbitals & Density   |
            |                         +-----------------+-----------------+
            |                                           |
            |                                           v
            |                                 /-------------------\
            +<------- [ No: Update ] --------<    Converged?       >
                                              \-------------------/
                                                        |
                                                     [ Yes ]
                                                        v
                                      +-----------------------------------+
                                      | Output Total Energy: E(R-B3LYP)   |
                                      +-----------------------------------+
<pre>

## 3. Code Dive-in

### Required Imports

The following libraries and modules are imported for the workflow:

- **`subprocess`**: Used to run the GAMESS input files according to their respective configurations.
- **`parser`**: Used to read and extract useful data from the GAMESS output files.
- **`config`**: Used to import the saved directories defined through the machine-specific `.env` variables.
- **`tabulate_molecules`**:  Provides functions for tabulating the parsed output.

NOTE: Please ensure that the molecular input files are also present in the respective GAMESS Installation directory for the workflow to compute.

In [1]:
# Required Imports
import subprocess
from pathlib import Path
import parser
from config import GAMESS_DIR, OUTPUT_DIR
from tabulate_molecules import (
    create_results_table_water,
    create_results_table_ethanol,
    create_results_table_methane,
    create_results_table_formaldehyde,
    create_results_table_hcn
)

Initializing the parameters for the input file runs.

In [2]:
# Configuration required for GAMESS
RUNGMS = GAMESS_DIR / "rungms.bat"
GAMESS_VERSION = "2023.R1.intel"
NCORES = "1"

Different molecules are selected with different properties. 

The configurations regarding Functionals also vary. 

Five different Functionals are used for each molecule. 

But the Basis configuration remain the same.

In [3]:
# Water
water_SVWN = "water_SVWN_LDA_631Gdp"
water_PBE = "water_PBE_GGA_631Gdp"
water_revTPSS = "water_revTPSS_metaGGA_631Gdp"
water_B3LYP = "water_B3LYP_hybridGGA_631Gdp"
water_M06 = "water_M06_hybridMetaGGA_631Gdp"


# Ethanol
ethanol_SVWN = "ethanol_SVWN_LDA_631Gdp"
ethanol_PBE = "ethanol_PBE_GGA_631Gdp"
ethanol_revTPSS = "ethanol_revTPSS_metaGGA_631Gdp"
ethanol_B3LYP = "ethanol_B3LYP_hybridGGA_631Gdp"
ethanol_M06 = "ethanol_M06_hybridMetaGGA_631Gdp"


# Methane
methane_SVWN = "methane_SVWN_LDA_631Gdp"
methane_PBE = "methane_PBE_GGA_631Gdp"
methane_revTPSS = "methane_revTPSS_metaGGA_631Gdp"
methane_B3LYP = "methane_B3LYP_hybridGGA_631Gdp"
methane_M06 = "methane_M06_hybridMetaGGA_631Gdp"


# Formaldehyde
formaldehyde_SVWN = "formaldehyde_SVWN_LDA_631Gdp"
formaldehyde_PBE = "formaldehyde_PBE_GGA_631Gdp"
formaldehyde_revTPSS = "formaldehyde_revTPSS_metaGGA_631Gdp"
formaldehyde_B3LYP = "formaldehyde_B3LYP_hybridGGA_631Gdp"
formaldehyde_M06 = "formaldehyde_M06_hybridMetaGGA_631Gdp"


# Hydrogen Cyanide
hcn_SVWN = "hcn_SVWN_LDA_631Gdp"
hcn_PBE = "hcn_PBE_GGA_631Gdp"
hcn_revTPSS = "hcn_revTPSS_metaGGA_631Gdp"
hcn_B3LYP = "hcn_B3LYP_hybridGGA_631Gdp"
hcn_M06 = "hcn_M06_hybridMetaGGA_631Gdp"

Major function for runnning GAMESS. Acts as a wrapper.

In [4]:
# Function to run the GAMESS input files as jobs
def run_gamess(job_name):

    result = subprocess.run(
        [
            str(GAMESS_DIR / "rungms.bat"),
            job_name,
            "2023.R1.intel",
            "1"
        ],
        cwd=GAMESS_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=True
    )

    output_text = result.stdout.replace("\r", "\n")

    # Save GAMESS output as TXT files
    output_file = OUTPUT_DIR / f"{job_name}.txt"
    output_file.write_text(
        output_text,
        encoding="utf-8"
    )

    return output_text

GAMESS function calls.

In [5]:
# Calling all GAMESS jobs

# Water
run_gamess(water_SVWN)
run_gamess(water_PBE)
run_gamess(water_revTPSS)
run_gamess(water_B3LYP)
run_gamess(water_M06)

# Ethanol
run_gamess(ethanol_SVWN)
run_gamess(ethanol_PBE)
run_gamess(ethanol_revTPSS)
run_gamess(ethanol_B3LYP)
run_gamess(ethanol_M06)

# Methane
run_gamess(methane_SVWN)
run_gamess(methane_PBE)
run_gamess(methane_revTPSS)
run_gamess(methane_B3LYP)
run_gamess(methane_M06)

# Formaldehyde
run_gamess(formaldehyde_SVWN)
run_gamess(formaldehyde_PBE)
run_gamess(formaldehyde_revTPSS)
run_gamess(formaldehyde_B3LYP)
run_gamess(formaldehyde_M06)

# Hydrogen Cyanide
run_gamess(hcn_SVWN)
run_gamess(hcn_PBE)
run_gamess(hcn_revTPSS)
run_gamess(hcn_B3LYP)
run_gamess(hcn_M06);

Parsing the output files by extracting the most important data and storing them as `.json` files.

In [6]:
# Calling parser function to retrieve useful data as JSON files

# Water
parser.parse_gamess(water_SVWN)
parser.parse_gamess(water_PBE)
parser.parse_gamess(water_revTPSS)
parser.parse_gamess(water_B3LYP)
parser.parse_gamess(water_M06)

# Ethanol
parser.parse_gamess(ethanol_SVWN)
parser.parse_gamess(ethanol_PBE)
parser.parse_gamess(ethanol_revTPSS)
parser.parse_gamess(ethanol_B3LYP)
parser.parse_gamess(ethanol_M06)

# Methane
parser.parse_gamess(methane_SVWN)
parser.parse_gamess(methane_PBE)
parser.parse_gamess(methane_revTPSS)
parser.parse_gamess(methane_B3LYP)
parser.parse_gamess(methane_M06)

# Formaldehyde
parser.parse_gamess(formaldehyde_SVWN)
parser.parse_gamess(formaldehyde_PBE)
parser.parse_gamess(formaldehyde_revTPSS)
parser.parse_gamess(formaldehyde_B3LYP)
parser.parse_gamess(formaldehyde_M06)

# Hydrogen Cyanide
parser.parse_gamess(hcn_SVWN)
parser.parse_gamess(hcn_PBE)
parser.parse_gamess(hcn_revTPSS)
parser.parse_gamess(hcn_B3LYP)
parser.parse_gamess(hcn_M06);

Tabulating each molecule's data.

In [7]:
# Tabulate the parsed data

df_water = create_results_table_water()
df_water

,Job,Method,Basis,SCF Converged,SCF Iterations,Total Energy (Hartree),Nuclear Repulsion (Hartree),Exchange-Correlation (Hartree),Electron Number
0,water_B3LYP_hybridGGA_631Gdp,B3LYP,"6-61G(d,p)",True,15,-76.382488,9.189534,-7.529789,10.0
1,water_M06_hybridMetaGGA_631Gdp,M06,"6-61G(d,p)",True,16,-76.385869,9.189534,-6.908843,10.0
2,water_PBE_GGA_631Gdp,PBE,"6-61G(d,p)",True,15,-76.332785,9.189534,-9.265551,10.0
3,water_revTPSS_metaGGA_631Gdp,REVTPSS,"6-61G(d,p)",True,15,-76.461726,9.189534,-9.404545,10.0
4,water_SVWN_LDA_631Gdp,SVWN,"6-61G(d,p)",True,15,-75.854679,9.189534,-8.773627,10.0


In [8]:
df_ethanol = create_results_table_ethanol()
df_ethanol

,Job,Method,Basis,SCF Converged,SCF Iterations,Total Energy (Hartree),Nuclear Repulsion (Hartree),Exchange-Correlation (Hartree),Electron Number
0,ethanol_B3LYP_hybridGGA_631Gdp,B3LYP,"6-61G(d,p)",True,18,-154.948258,82.010751,-17.472718,25.999943
1,ethanol_M06_hybridMetaGGA_631Gdp,M06,"6-61G(d,p)",True,19,-154.938726,82.010751,-16.004257,25.999994
2,ethanol_PBE_GGA_631Gdp,PBE,"6-61G(d,p)",True,18,-154.839772,82.010751,-21.494274,25.999941
3,ethanol_revTPSS_metaGGA_631Gdp,REVTPSS,"6-61G(d,p)",True,19,-155.147628,82.010751,-21.838523,25.999995
4,ethanol_SVWN_LDA_631Gdp,SVWN,"6-61G(d,p)",True,18,-153.729267,82.010751,-20.328260,25.999937


In [10]:
df_methane = create_results_table_methane()
df_methane

,Job,Method,Basis,SCF Converged,SCF Iterations,Total Energy (Hartree),Nuclear Repulsion (Hartree),Exchange-Correlation (Hartree),Electron Number
0,methane_B3LYP_hybridGGA_631Gdp,B3LYP,"6-61G(d,p)",True,14,-40.488037,13.472036,-5.560739,10.000469
1,methane_M06_hybridMetaGGA_631Gdp,M06,"6-61G(d,p)",True,15,-40.479086,13.472036,-5.088585,10.000031
2,methane_PBE_GGA_631Gdp,PBE,"6-61G(d,p)",True,14,-40.449689,13.472036,-6.835075,10.000480
3,methane_revTPSS_metaGGA_631Gdp,REVTPSS,"6-61G(d,p)",True,14,-40.553348,13.472036,-6.952598,10.000030
4,methane_SVWN_LDA_631Gdp,SVWN,"6-61G(d,p)",True,15,-40.102273,13.472036,-6.463659,10.000503


In [ ]:
df_hcn = create_results_table_hcn()
df_hcn

,Job,Method,Basis,SCF Converged,SCF Iterations,Total Energy (Hartree),Nuclear Repulsion (Hartree),Exchange-Correlation (Hartree),Electron Number
0,hcn_B3LYP_hybridGGA_631Gdp,B3LYP,"6-61G(d,p)",True,17,-93.372848,23.878822,-10.137255,14.0
1,hcn_M06_hybridMetaGGA_631Gdp,M06,"6-61G(d,p)",True,19,-93.359848,23.878822,-9.279376,14.0
2,hcn_PBE_GGA_631Gdp,PBE,"6-61G(d,p)",True,18,-93.309086,23.878822,-12.478423,14.0
3,hcn_revTPSS_metaGGA_631Gdp,REVTPSS,"6-61G(d,p)",True,18,-93.484128,23.878822,-12.670160,14.0
4,hcn_SVWN_LDA_631Gdp,SVWN,"6-61G(d,p)",True,17,-92.611884,23.878822,-11.753733,14.0


In [ ]:
df_formald = create_results_table_formaldehyde()
df_formald

,Job,Method,Basis,SCF Converged,SCF Iterations,Total Energy (Hartree),Nuclear Repulsion (Hartree),Exchange-Correlation (Hartree),Electron Number
0,formaldehyde_B3LYP_hybridGGA_631Gdp,B3LYP,"6-61G(d,p)",True,17,-114.443889,31.255737,-11.829652,16.000036
1,formaldehyde_M06_hybridMetaGGA_631Gdp,M06,"6-61G(d,p)",True,20,-114.437331,31.255737,-10.838578,15.999999
2,formaldehyde_PBE_GGA_631Gdp,PBE,"6-61G(d,p)",True,18,-114.369508,31.255737,-14.556685,16.000037
3,formaldehyde_revTPSS_metaGGA_631Gdp,REVTPSS,"6-61G(d,p)",True,19,-114.576583,31.255737,-14.785365,15.999999
4,formaldehyde_SVWN_LDA_631Gdp,SVWN,"6-61G(d,p)",True,19,-113.587572,31.255737,-13.743020,16.000038
